# Exercise 4: Pivot translation comparison

*Machine Translation · version 0.1.0*

> For the language pair Uzbek→Kyrgyz, implement two translation strategies using NLLB-200: (a) direct translation with codes `uzn_Latn → kir_Cyrl`; (b) pivot through Turkish (`uzn_Latn → tur_Latn`, then `tur_Latn → kir_Cyrl`). With TurkicNLP, (a) is `Pipeline("uzb", processors=["translate"], translate_tgt_lang="kir")` and (b) chains a `uzb→tur` and a `tur→kir` pipeline. Translate the same 50 held-out sentences (for example from the X-WMT Uzbek–Kyrgyz test set) under matched decoding settings; evaluate each strategy with chrF++, record added latency and inspect error propagation. Which approach scores higher on this sample? Use paired examples to investigate possible causes rather than treating the score difference as an explanation.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" torch transformers sentencepiece sacrebleu requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`). NLLB-200 distilled 600M is loaded from `NLLB_PATH` (a local copy) or from the Hugging Face Hub at the revision pinned in the book; it runs in float32 on the CPU, about 1.7 s per sentence on four cores. Translations are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so an interrupted run resumes where it stopped. Test data come from the commit-pinned TIL release (X-WMT test sets, MIT licence); the X-WMT files are tokenised, and the code removes the spaces around punctuation before scoring (a heuristic, see `detok`). Scores use SacreBLEU with its default settings: BLEU with the `13a` tokeniser, chrF, and chrF++ (`word_order=2`).

The 50 sentences are the first 50 of the X-WMT Uzbek–Kyrgyz test set. All legs use the same greedy decoding with a 256-token ceiling. A Russian pivot is added as a third strategy. Generation time is the sum of the batch times recorded with the cached translations, so it is reported correctly even when a run resumes from the cache.

In [ ]:
import collections, hashlib, os, re, statistics, time
from pathlib import Path
import requests

# Commit-pinned TIL release files (X-WMT test sets and the corpus README); MIT licence.
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation) so references match raw MT output."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)       # Sialkot ’ taki -> Sialkot’taki
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt(src, tgt, n=None):
    """The first n sentence pairs of the X-WMT test set for src-tgt (X-WMT language codes)."""
    s = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{src}").read_text(encoding="utf-8").splitlines()
    t = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{tgt}").read_text(encoding="utf-8").splitlines()
    assert len(s) == len(t), (src, tgt)
    pairs = [(detok(a), detok(b)) for a, b in zip(s, t) if a.strip() and b.strip()]
    return pairs[:n] if n else pairs

NLLB_CODE = {"az": "azj_Latn", "ba": "bak_Cyrl", "kk": "kaz_Cyrl", "ky": "kir_Cyrl", "tr": "tur_Latn",
             "uz": "uzn_Latn", "en": "eng_Latn", "ru": "rus_Cyrl"}
NLLB = {}

def nllb():
    """NLLB-200 distilled 600M: from NLLB_PATH (a local copy) or the pinned Hub revision; float32, CPU."""
    if not NLLB:
        import torch
        from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
        path = os.environ.get("NLLB_PATH", "facebook/nllb-200-distilled-600M")
        rev = None if Path(path).exists() else "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
        NLLB["tok"] = NllbTokenizer.from_pretrained(path, revision=rev)
        NLLB["model"] = AutoModelForSeq2SeqLM.from_pretrained(path, revision=rev, torch_dtype=torch.float32).eval()
        NLLB["torch"] = torch
        torch.set_num_threads(os.cpu_count() or 1)
    return NLLB["tok"], NLLB["model"]

CACHE_FILE = CACHE / "nllb-translations.jsonl"       # translations are cached, so a run can resume
LAST_SECONDS = 0.0

def _cached():
    import json
    out = {}
    if CACHE_FILE.exists():
        for line in CACHE_FILE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[tuple(r["key"])] = (r["out"], r["capped"], r["sec"])
    return out

def translate(sents, src, tgt, beams=1, batch=8, max_new=256):
    """Translate with explicit source and target language tokens (greedy unless beams > 1).
    Returns (outputs, number of outputs that reached the generation ceiling without an
    end-of-sequence token). Results are cached per sentence in CACHE_FILE, with the generation
    time of their batch divided by its size; LAST_SECONDS holds the total for the last call."""
    global LAST_SECONDS
    import json
    cache = _cached()
    key = lambda s: [src, tgt, beams, max_new, s]
    todo = [s for s in dict.fromkeys(sents) if tuple(key(s)) not in cache]
    if todo:
        tok, model = nllb()
        tok.src_lang = NLLB_CODE[src]
        CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
        with NLLB["torch"].inference_mode(), CACHE_FILE.open("a", encoding="utf-8") as f:
            for i in range(0, len(todo), batch):
                chunk = todo[i:i + batch]
                t0 = time.time()
                enc = tok(chunk, return_tensors="pt", padding=True)
                gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_CODE[tgt]),
                                     num_beams=beams, do_sample=False, max_new_tokens=max_new)
                sec = (time.time() - t0) / len(chunk)
                for s, row, text in zip(chunk, gen, tok.batch_decode(gen, skip_special_tokens=True)):
                    capped = int(row[-1] != tok.eos_token_id and row[-1] != tok.pad_token_id)
                    cache[tuple(key(s))] = (text, capped, sec)
                    f.write(json.dumps({"key": key(s), "out": text, "capped": capped, "sec": sec}, ensure_ascii=False) + "\n")
                f.flush()
    res = [cache[tuple(key(s))] for s in sents]
    LAST_SECONDS = sum(r[2] for r in res)
    return [r[0] for r in res], sum(r[1] for r in res)

def metrics(hyps, refs):
    """Corpus BLEU (13a), chrF and chrF++ with SacreBLEU, rounded, plus the chrF++ signature."""
    from sacrebleu.metrics import BLEU, CHRF
    bleu, chrf, chrfpp = BLEU(tokenize="13a"), CHRF(), CHRF(word_order=2)
    r = {"BLEU": bleu.corpus_score(hyps, [refs]).score, "chrF": chrf.corpus_score(hyps, [refs]).score,
         "chrF++": chrfpp.corpus_score(hyps, [refs]).score}
    return {k: round(v, 2) for k, v in r.items()}, str(chrfpp.get_signature())

In [ ]:
pairs = xwmt("uz", "ky", 50)                      # the first 50 X-WMT Uzbek-Kyrgyz test sentences
src, refs = [s for s, _ in pairs], [t for _, t in pairs]
direct, c1 = translate(src, "uz", "ky"); t_direct = LAST_SECONDS
turkish, c2 = translate(src, "uz", "tr"); t_pivot = LAST_SECONDS
pivot, c3 = translate(turkish, "tr", "ky"); t_pivot += LAST_SECONDS
russian, c4 = translate(src, "uz", "ru"); t_ru = LAST_SECONDS
pivot_ru, c5 = translate(russian, "ru", "ky"); t_ru += LAST_SECONDS
for name, hyp, sec in [("direct uzn->kir", direct, t_direct), ("pivot via tur_Latn", pivot, t_pivot), ("pivot via rus_Cyrl", pivot_ru, t_ru)]:
    print(f"{name:20} {metrics(hyp, refs)[0]}  generation time {sec:.0f} s")
print("outputs at the length ceiling:", c1, c2 + c3, c4 + c5)
from sacrebleu.metrics import CHRF
s = CHRF(word_order=2)
gain = [s.sentence_score(p, [r]).score - s.sentence_score(d, [r]).score for d, p, r in zip(direct, pivot, refs)]
print(f"Turkish pivot better on {sum(g > 0 for g in gain)} sentences, worse on {sum(g < 0 for g in gain)}, equal on {sum(g == 0 for g in gain)}")
worst = min(range(len(gain)), key=lambda i: gain[i])
print("\nlargest pivot loss:\n  src  :", src[worst], "\n  ref  :", refs[worst], "\n  direct:", direct[worst],
      "\n  tur  :", turkish[worst], "\n  pivot:", pivot[worst])

## Worked solution

Output of `experiments/ch10_exercises/run_ch10_exercises.py` (NLLB-200 distilled 600M, CPU, float32):

```
direct uzn->kir      {'BLEU': 12.81, 'chrF': 45.69, 'chrF++': 41.15}  generation time 145 s
pivot via tur_Latn   {'BLEU': 9.68, 'chrF': 40.98, 'chrF++': 36.46}  generation time 287 s
pivot via rus_Cyrl   {'BLEU': 11.43, 'chrF': 45.08, 'chrF++': 40.32}  generation time 215 s
outputs at the length ceiling: 1 2 0
Turkish pivot better on 15 sentences, worse on 35, equal on 0

largest pivot loss:
  src  : Shiforkorlarning ommaviy ish tashlashiga rahnamolik qilgani uchun o’g’irlangan va aytishlaricha qiynalgan Zimbabvedagi shifokor payshanba kuni kechqurun davlatni, uning tibbiy guruhi aytishicha Janubiy Afrikada’ shoshilinch’ davolanish maqsadida tark etdi. 
  ref  : Дарыгерлердин иш таштоосун уюштургандыгы үчүн уурдалган жана кыйноого дуушар болгон Зимбабвелик дарыгер, анын медициналык тобу Түштүк Африка Республикасында " шашылыш " дарылоо үчүн, бейшемби күнү кечинде өлкөдөн чыгып кетти. 
  direct: Зимбабведеги дарыгер медициналык кызматкерлердин массалык түрдө токтотулушуна жол бергени үчүн уурдалып, кыйналып жүргөнүн айтышкан. Бейшемби күнү кечинде өлкөдөн чыгып кеткен. 
  tur  : Bir doktor, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun, bir doktorun acil bir tedavi amacıyla Güney Afrika'da, bir hastane, bir hastane, bir hastane, bir hastane, bir hastane, bir hastane, bir hast 
  pivot: Бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер, бир дарыгер
```

**Direct translation wins.** Direct Uzbek→Kyrgyz reaches chrF++ 41.15 (BLEU 12.81). The Russian pivot comes close (40.32), and the Turkish pivot is clearly worse (36.46). The pivots also cost time: the Turkish route needs about twice the generation time of the direct one (287 s against 145 s for 50 sentences on this CPU), the Russian route about 1.5 times.

**Error propagation.** The Turkish pivot is better on 15 sentences and worse on 35. Its largest loss shows how pivot errors compound: on one long sentence the Uzbek→Turkish leg degenerates into a repetition loop (*bir doktorun, bir doktorun, …*) that runs until the length ceiling, and the Turkish→Kyrgyz leg faithfully translates the loop. The direct translation of the same sentence is incomplete (it drops the clause about South Africa) but adequate in its first part. Two of the three outputs that reached the length ceiling come from the Turkish route.

**Why.** NLLB-200 already translates Uzbek→Kyrgyz directly, so a pivot adds a second decoding step and its errors without adding information. A pivot can help when the direct direction is much weaker than both legs. Whether that holds for a pair has to be measured: here it does not, and the TIL release also lists 119,946 Uzbek–Kyrgyz training pairs, more than for many pairs with a Turkish leg.

## Your turn — extension

1. Repeat with beam search (4 beams) for all legs. Does the ranking change?
2. Try Kazakh→Kyrgyz, where TIL has only 12,216 training pairs.